## Importing Libraries

In [1]:
import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn import set_config

import warnings

## Importing Dataset

In [2]:
data = pd.read_csv("laptop_prices.csv")

# To hide unncessary warnings and make notebook look clean
warnings.filterwarnings('ignore')

## Feature Engineering

In [3]:
# Calculate PPI (Pixels Per Inch) using screen resolution and screen size:
data['PPI'] = (np.sqrt((data['ScreenW']**2) + (data['ScreenH']**2))) / data['Inches']

# Drop resolution columns after creating PPI to avoid redundancy:
data.drop(columns=['ScreenW','ScreenH', 'Inches'], inplace=True)

In [4]:
# Created new columns for each storage type:
data['SSD'] = 0
data['HDD'] = 0
data['Flash_Storage'] = 0
data['Hybrid'] = 0

# Assign primary storage capacity to the corresponding storage type:
data.loc[data['PrimaryStorageType'] == 'SSD', 'SSD'] = data['PrimaryStorage']
data.loc[data['PrimaryStorageType'] == 'HDD', 'HDD'] = data['PrimaryStorage'] 
data.loc[data['PrimaryStorageType'] == 'Flash Storage', 'Flash_Storage'] = data['PrimaryStorage']
data.loc[data['PrimaryStorageType'] == 'Hybrid', 'Hybrid'] = data['PrimaryStorage']

# Add secondary storage capacity to the corresponding storage type:
data.loc[data['SecondaryStorageType'] == 'SSD', 'SSD'] += data['SecondaryStorage']
data.loc[data['SecondaryStorageType'] == 'HDD', 'HDD'] += data['SecondaryStorage']
data.loc[data['SecondaryStorageType'] == 'Flash Storage', 'Flash_Storage'] += data['SecondaryStorage']
data.loc[data['SecondaryStorageType'] == 'Hybrid', 'Hybrid'] += data['SecondaryStorage']

# Drop old storage columns after extracting useful information:
data.drop(columns=['PrimaryStorage', 'SecondaryStorage', 'PrimaryStorageType', 'SecondaryStorageType'], inplace=True)

In [5]:
# Created a new column which will store CPU tiers:
data['CPU_tier'] = 'other'

# Extarcting CPU tiers from CPU models:
data.loc[data['CPU_model'].str.contains('i7', case=False, na=False),'CPU_tier'] = 'i7'
data.loc[data['CPU_model'].str.contains('i5', case=False, na=False),'CPU_tier'] = 'i5'
data.loc[data['CPU_model'].str.contains('i3', case=False, na=False),'CPU_tier'] = 'i3'
data.loc[data['CPU_model'].str.contains('Pentium', case=False, na=False),'CPU_tier'] = 'Pentium'
data.loc[data['CPU_model'].str.contains('Celeron', case=False, na=False),'CPU_tier'] = 'Celeron'
data.loc[data['CPU_model'].str.contains('AMD', case=False, na=False),'CPU_tier'] = 'AMD'
data.loc[data['CPU_model'].str.contains('Ryzen', case=False, na=False),'CPU_tier'] = 'AMD'

# Drop original CPU_model column after extracting useful information:
data.drop(columns=['CPU_model'], inplace=True)

In [6]:
# Created a new column which will store GPU tiers:
data['GPU_series'] = 'other'

# Extarcting GPU series from GPU models:
data.loc[data['GPU_model'].str.contains('GTX', case=False, na=False), 'GPU_series'] = 'GTX'
data.loc[data['GPU_model'].str.contains('MX', case=False, na=False), 'GPU_series'] = 'MX'
data.loc[data['GPU_model'].str.contains('GT', case=False, na=False), 'GPU_series'] = 'GT'
data.loc[data['GPU_model'].str.contains('UHD', case=False, na=False), 'GPU_series'] = 'Intel UHD'
data.loc[data['GPU_model'].str.contains('HD', case=False, na=False), 'GPU_series'] = 'Intel HD'
data.loc[data['GPU_model'].str.contains('Iris', case=False, na=False), 'GPU_series'] = 'Intel Iris'
data.loc[data['GPU_model'].str.contains('Radeon', case=False, na=False), 'GPU_series'] = 'Radeon'

# Drop original GPU_model column after extracting useful information:
data.drop(columns=['GPU_model'], inplace=True)

In [7]:
# Created a new Brand feature by grouping major laptop companies, others labeled as 'other:
data['Brand'] = 'other'

brands = ['Dell', 'Lenovo', 'HP', 'Asus', 'Acer', 'MSI', 'Toshiba', 'Apple']

data.loc[data['Company'].isin(brands), 'Brand'] = data['Company'] 

# Drop original Company column after extracting the brand information and also Product column:
data.drop(columns=['Company', 'Product'], inplace=True)

## Train_Test_Split

In [8]:
X = data.drop(columns=['Price_euros'])
y = np.log(data['Price_euros'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)

## Encoding

In [9]:
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

ohe.fit(X_train)

X_train_transform = ohe.transform(X_train)
X_test_transform = ohe.transform(X_test)

## Model Training and Testing
------------------------------
### 1. Linear Regression

In [10]:
lr = LinearRegression()

lr.fit(X_train_transform, y_train)

y_pred = lr.predict(X_test_transform)

print(f"Accuracy: {r2_score(y_test, y_pred)*100}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred))}")

n = X_train.shape[0]
k = X_train.shape[1]
r2 = r2_score(y_test,y_pred)
adj_r2 = 1 - (((1-r2)*(n-1))/(n-k-1))
print(f"Adjusted R2 Score: {adj_r2}")

Accuracy: 85.8558746462437
Mean Absolute Error: 0.1771889326360324
Root Mean Squared Error: 0.22693579912619985
Adjusted R2 Score: 0.8558713626452233


## 2. Ridge Regression

In [17]:
ridge = Ridge()

ridge.fit(X_train_transform, y_train)

y_pred = ridge.predict(X_test_transform)

print(f"Accuracy: {r2_score(y_test, y_pred)*100}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred))}")

n = X_train.shape[0]
k = X_train.shape[1]
r2 = r2_score(y_test,y_pred)
adj_r2 = 1 - (((1-r2)*(n-1))/(n-k-1))
print(f"Adjusted R2 Score: {adj_r2}")

Accuracy: 88.69596486321952
Mean Absolute Error: 0.16177848494773261
Root Mean Squared Error: 0.20287648525472948
Adjusted R2 Score: 0.8848118819562069


## 3. Lasso Regression

In [18]:
lasso = Lasso(alpha=0.0001)

lasso.fit(X_train_transform, y_train)

y_pred = lasso.predict(X_test_transform)

print(f"Accuracy: {r2_score(y_test, y_pred)*100}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred))}")

n = X_train.shape[0]
k = X_train.shape[1]
r2 = r2_score(y_test,y_pred)
adj_r2 = 1 - (((1-r2)*(n-1))/(n-k-1))
print(f"Adjusted R2 Score: {adj_r2}")

Accuracy: -0.06625727995215325
Mean Absolute Error: 0.4954159355109173
Root Mean Squared Error: 0.6036132870580072
Adjusted R2 Score: -0.01967516168271244


## 4. Elastic-Net Regression

In [19]:
en = ElasticNet()

en.fit(X_train_transform, y_train)

y_pred = en.predict(X_test_transform)

print(f"Accuracy: {r2_score(y_test, y_pred)*100}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred))}")

n = X_train.shape[0]
k = X_train.shape[1]
r2 = r2_score(y_test,y_pred)
adj_r2 = 1 - (((1-r2)*(n-1))/(n-k-1))
print(f"Adjusted R2 Score: {adj_r2}")

Accuracy: -0.06625727995215325
Mean Absolute Error: 0.4954159355109173
Root Mean Squared Error: 0.6036132870580072
Adjusted R2 Score: -0.01967516168271244


## Building Pipeline

In [14]:
num_cols = ['Ram', 'Weight', 'CPU_freq', 'PPI', 'SSD', 'HDD', 'Flash_Storage', 'Hybrid']
cat_cols = ['TypeName', 'OS', 'Screen', 'Touchscreen', 'IPSpanel', 'RetinaDisplay', 'CPU_company', 'GPU_company', 'CPU_tier', 'GPU_series', 'Brand']

handling_num_values = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

handling_cat_cols = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessing = ColumnTransformer(transformers=[
    ('num', handling_num_values, num_cols),
    ('cat', handling_cat_cols, cat_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', Ridge())
])

param_grid = {'model__alpha' : [0.1, 0.5, 1, 2, 5, 10, 20, 50, 100]}
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv = 5,
    scoring = 'r2',
    n_jobs = -1
)

model = TransformedTargetRegressor(
    regressor=grid,
    func=np.log1p,
    inverse_func=np.expm1
)

In [15]:
set_config(display='diagram')
model

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",GridSearchCV(... scoring='r2')
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed ou

In [16]:
X = data.drop(columns=['Price_euros'])
y = data['Price_euros']

scores = cross_val_score(model, X, y, scoring='r2', cv=5)

print(f"Accuracy of my model = {np.mean(scores)*100}%")

Accuracy of my model = 66.64900132696233%


In [40]:
model.fit(X, y)

best_alpha = model.regressor_.best_params_['model__alpha']
print("Best Alpha:", best_alpha)

Best Alpha: 0.1
